## 3 Notebook - Camada Silver

Em um segundo notebook (Silver), crie o banco de dados da camada Silver e implemente as transformações de limpeza e padronização. Nenhuma alteração será realizada nas tabelas da camada bronze. Todos os nomes de colunas devem estar em português e a tipagem deve estar correta.

---

## Regras de Negócio e Transformações Exigidas

### 1. `silver.dim_consumidores` (origem: `tb_customers`)
- **Mapeamento:**
  - `customer_id` -> `id_consumidor`
  - `customer_zip_code_prefix` -> `prefixo_cep`
  - `customer_name` -> `nome_consumidor`
  - `customer_city` -> `cidade`
  - `customer_state` -> `estado`

- **Deduplicação Sênior:**
  - A coluna `id_consumidor` não deve conter valores duplicados.
  - Utilizar *Window Functions* particionadas pelo ID e ordenadas de forma decrescente pelo `timestamp_ingestion`, mantendo apenas o registro mais recente.

- **Padronização:**
  - Nomes de estado e cidade devem estar em **UPPER CASE**.

---

### 2. `silver.fat_pedidos` (origem: `tb_orders`)
- **Conversão de status (EN → PT):**
  - delivered → entregue  
  - canceled → cancelado  
  - shipped → enviado  
  - processing → processando  
  - invoiced → faturado  
  - unavailable → indisponível  
  - created → criado  
  - approved → aprovado  

- **Novas colunas derivadas:**
  - `tempo_entrega_dias`: diferença entre data de entrega real e data de compra  
  - `tempo_entrega_estimado_dias`: diferença entre data estimada e data de compra  
  - `diferenca_entrega_dias`: diferença entre tempo real e estimado  
  - `entrega_no_prazo`: indicador textual ("Sim", "Não", "Não Entregue")  

---

### 3. `silver.fat_itens_pedidos` (origem: `tb_order_items`)
- **Mapeamento:**
  - `order_id` -> `id_pedido`
  - `order_item_id` -> `id_item`
  - `product_id` -> `id_produto`
  - `seller_id` -> `id_vendedor`
  - `price` -> `preco_BRL`
  - `freight_value` -> `preco_frete`

---

### 4. `silver.fat_pagamentos_pedidos` (origem: `tb_order_payments`)
- **Conversão de tipo de pagamento (EN → PT):**
  - credit_card → Cartão de Crédito  
  - boleto → Boleto  
  - voucher → Voucher  
  - debit_card → Cartão de Débito  
  - not_defined → Não Definido  

---

### 5. `silver.fat_avaliacoes_pedidos` (origem: `tb_order_reviews`)
- **Limpeza de dados:**
  - Remover registros com pedido inválido  
  - Remover datas de comentário no futuro  

- **Tolerância a falhas:**
  - Utilizar `try_to_timestamp` para evitar erros com dados inválidos  

- **Tratamento de nulos:**
  - Comentários vazios → "Sem comentário"  
  - Títulos vazios → "Sem título"  

- **Mapeamento:**
  - `review_id` -> `id_avaliacao`
  - `order_id` -> `id_pedido`
  - `review_score` -> `nota_avaliacao`
  - `review_comment_title` -> `titulo_avaliacao`
  - `review_comment_message` -> `comentario_avaliacao`
  - `review_creation_date` -> `data_criacao_avaliacao`
  - `review_answer_timestamp` -> `data_resposta_avaliacao`

---

### 6. `silver.dim_produtos` (origem: `tb_products`)
- **Deduplicação Sênior:**
  - Aplicar Window Function baseada em `timestamp_ingestion`

- **Mapeamento:**
  - `product_id` -> `id_produto`
  - `product_name` -> `nome_produto`
  - `product_category_name` -> `categoria_produto`
  - Dimensões:
    - `peso_produto_gramas`
    - `comprimento_centimetros`
    - `altura_centimetros`
    - `largura_centimetros`
    - `quantidade_fotos`
    - `tamanho_nome_produto`
    - `tamanho_descricao_produto`

---

### 7. `silver.dim_vendedores` (origem: `tb_sellers`)
- **Deduplicação Sênior:**
  - Aplicar Window Function baseada em `timestamp_ingestion`

- **Padronização:**
  - Cidade e estado em **UPPER CASE**

- **Mapeamento:**
  - `seller_id` -> `id_vendedor`
  - `seller_name` -> `nome_vendedor`
  - `seller_zip_code_prefix` -> `prefixo_cep`
  - `seller_city` -> `cidade`
  - `seller_state` -> `estado`

---

### 8. `silver.dim_categoria_produtos_traducao`
- Mapear nomes de produtos:
  - `nome_produto_pt`
  - `nome_produto_en`

---

### 9. `silver.dim_cotacao_dolar`
- Criar calendário contínuo (incluindo finais de semana)
- Preencher valores ausentes com a cotação da sexta-feira anterior
- Utilizar:
  - `last(ignorenulls=True)` com Window Function

---

## Tabela Final Silver

### `silver.fat_pedido_total`
- **Join de:**
  - `fat_pedidos`
  - `fat_pagamentos_pedidos`
  - `dim_cotacao_dolar`

- **Campos finais:**
  - `id_pedido`
  - `id_consumidor`
  - `status`
  - `valor_total_pago_brl`
  - `valor_total_pago_usd`
  - `data_pedido`

- **Regra obrigatória:**
  - Arredondar valores financeiros para **2 casas decimais**

---

## Otimização Física (Delta Optimize)

Ao final do notebook da Camada Silver, executar:

- `OPTIMIZE` com `ZORDER`
- Indexação por:
  - `id_pedido`
  - `data_pedido`

Objetivo: garantir alta performance analítica.

In [0]:
%sql
USE CATALOG medalhao;

CREATE SCHEMA IF NOT EXISTS silver;

USE SCHEMA silver;

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, row_number, upper, current_timestamp

# 1. LEITURA
df_bronze_customers = spark.read.table("bronze.tb_customers")

# 2. REGRA DE NEGÓCIO: DESDUPLICAÇÃO 
# Justificativa: Um cliente pode ter múltiplos registros na camada Bronze devido a reprocessamentos 
# ou atualizações de cadastro.
# Criei uma partição por 'customer_id' e ordenamos pelo 'timestamp_ingestion' decrescente para 
# garantir que o registro mais recente seja o escolhido.
window_spec = Window.partitionBy("customer_id").orderBy(col("timestamp_ingestion").desc())

df_silver_consumidores = (
    df_bronze_customers
    # Aplica a numeração de linha baseada na regra da linha "mais recente"
    .withColumn("rn", row_number().over(window_spec))
    # Filtro RN=1: Mantém apenas a última versão de cada cliente, eliminando duplicatas
    .filter(col("rn") == 1)
    # 3. REGRA DE NEGÓCIO: PADRONIZAÇÃO E TRADUÇÃO
    .select(
        col("customer_id").alias("id_consumidor"),
        col("customer_zip_code_prefix").alias("prefixo_cep"),
        col("customer_name").alias("nome_consumidor"),
        
        # Justificativa: Cidades e Estados costumam vir com variações de caixa (ex: "São Paulo" vs "SÃO PAULO").
        # O uso de upper() padroniza as chaves de busca e agrupamentos em relatórios de BI.
        upper(col("customer_city")).alias("cidade"),
        upper(col("customer_state")).alias("estado")
    )
)

# 4. REGRA DE NEGÓCIO: RASTREABILIDADE (CONTROLE DE AUDITORIA)
df_silver_consumidores = df_silver_consumidores.withColumn("timestamp_ingestion", current_timestamp())

# 5. CARGA FINAL
# Justificativa: O formato Delta garante transações ACID. O modo 'overwrite' com 'overwriteSchema' 
# é utilizado para garantir que a dimensão seja reconstruída com dados limpos
(
    df_silver_consumidores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true") 
    .saveAsTable("silver.dim_consumidores")
)

In [0]:
from pyspark.sql.functions import col, when, datediff, lower, current_timestamp

# 1. LEITURA
df_orders = spark.read.table("bronze.tb_orders")

# 2. REGRA DE NEGÓCIO: TRADUÇÃO DE STATUS
# Justificativa: Padronizar a linguagem dos status para português.
df_translated = df_orders.withColumn(
    "status",
    when(col("order_status") == "delivered", "entregue")
    .when(col("order_status") == "canceled", "cancelado")
    .when(col("order_status") == "shipped", "enviado")
    .when(col("order_status") == "processing", "processando")
    .when(col("order_status") == "invoiced", "faturado")
    .when(col("order_status") == "unavailable", "indisponível")
    .when(col("order_status") == "created", "criado")
    .when(col("order_status") == "approved", "aprovado")
    .otherwise(col("order_status"))
)

# 3. TRANSFORMAÇÃO E CÁLCULO DE MÉTRICAS
df_silver_pedidos = (
    df_translated.select(
        col("order_id").alias("id_pedido"),
        col("customer_id").alias("id_consumidor"),
        col("status"),
        col("order_purchase_timestamp").alias("data_compra"),
        col("order_approved_at").alias("data_aprovacao"),
        col("order_delivered_carrier_date").alias("data_entrega_transportadora"),
        col("order_delivered_customer_date").alias("data_entrega_cliente"),
        col("order_estimated_delivery_date").alias("data_estimada_entrega")
    )

    # Regra: Calcular quantos dias levou para o cliente receber o produto
    .withColumn("tempo_entrega_dias", datediff(col("data_entrega_cliente"), col("data_compra")))
    
    # Regra: Calcular a expectativa de entrega inicial dada ao cliente
    .withColumn("tempo_entrega_estimado_dias", datediff(col("data_estimada_entrega"), col("data_compra")))
    
    # Regra: Diferença entre o real e o estimado (Positivo (+) = Atraso, Negativo (-) = Adiantado)
    .withColumn("diferenca_entrega_dias", datediff(col("data_entrega_cliente"), col("data_estimada_entrega")))
    
    # Regra de Negócio SLA (Service Level Agreement): Classificação binária de pontualidade
    .withColumn("entrega_no_prazo", 
        when(col("data_entrega_cliente").isNull(), "Não Entregue")
        .when(col("diferenca_entrega_dias") <= 0, "Sim")
        .otherwise("Não")
    )
)

# 4. CONTROLE DE AUDITORIA
df_silver_pedidos = df_silver_pedidos.withColumn("timestamp_ingestion", current_timestamp())

# 5. SALVANDO A TABELA NA CAMADA SILVER
(
    df_silver_pedidos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fat_pedidos")
)

In [0]:
from pyspark.sql.functions import col, current_timestamp

# 1. LEITURA DOS DADOS
df_order_items = spark.read.table("bronze.tb_order_items")

# 2. SELEÇÃO E RENOMEAÇÃO
# Justificativa: Padronizar a linguagem dos status para português.
df_silver_itens_pedidos = (
    df_order_items.select(
        col("order_id").alias("id_pedido"),
        col("order_item_id").alias("id_item"),
        col("product_id").alias("id_produto"),
        col("seller_id").alias("id_vendedor"),
        col("price").alias("preco_BRL"),
        col("freight_value").alias("preco_frete")
    )
)

# 3. CONTROLE DE AUDITORIA
df_silver_itens_pedidos = df_silver_itens_pedidos.withColumn("timestamp_ingestion", current_timestamp())

# 4. SALVANDO A TABELA NA CAMADA SILVER
(
    df_silver_itens_pedidos.write
    .format("delta")
    .mode("overwrite")              
    .option("overwriteSchema", "true") 
    .saveAsTable("silver.fat_itens_pedidos")
)

In [0]:
from pyspark.sql.functions import col, when, current_timestamp

# 1. LEITURA DOS DADOS
df_payments = spark.read.table("bronze.tb_order_payments")

# 2. TRANSFORMAÇÃO E TRADUÇÃO (RULES)
df_silver_pagamentos = df_payments.select(
    col("order_id").alias("id_pedido"),
    col("payment_sequential").alias("sequencial_pagamento"),
    
    # Justificativa: Padronizar nomes de colunas para facilitar o JOIN com outras tabelas Silver
    when(col("payment_type") == "credit_card", "Cartão de Crédito")
    .when(col("payment_type") == "boleto", "Boleto")
    .when(col("payment_type") == "voucher", "Voucher")
    .when(col("payment_type") == "debit_card", "Cartão de Débito")
    .when(col("payment_type") == "not_defined", "Não Definido")
    .otherwise(col("payment_type")).alias("tipo_pagamento"),
    
    col("payment_installments").alias("parcelas_pagamento"),
    
    # Justificativa: Explicitando a unidade monetária em real para clareza em análises financeiras.
    col("payment_value").alias("valor_pagamento_brl")
)

# 3. CONTROLE DE AUDITORIA
df_silver_pagamentos = df_silver_pagamentos.withColumn("timestamp_ingestion", current_timestamp())

# 4. SALVANDO A TABELA NA CAMADA SILVER
(
    df_silver_pagamentos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fat_pagamentos_pedidos")
)

In [0]:
from pyspark.sql.functions import col, when, coalesce, lit, try_to_timestamp, current_timestamp, expr

# 1. LEITURA DE DADOS
df_reviews = spark.read.table("bronze.tb_order_reviews")

# 2. PRÉ-PROCESSAMENTO E FILTRAGEM
df_silver_avaliacoes = (
    df_reviews
    # Converte strings para timestamp; retorna nulo se o formato for inválido
    .withColumn("data_criacao_conv", try_to_timestamp(col("review_creation_date")))
    .withColumn("data_resposta_conv", try_to_timestamp(col("review_answer_timestamp")))
    
    # REGRA DE INTEGRIDADE: Remove reviews sem pedido vinculado e evita datas de criação inconsistentes como futuras
    .filter(
        (col("order_id").isNotNull()) & 
        (col("data_criacao_conv") <= current_timestamp())
    )
    
    # 3. SELEÇÃO E TRATAMENTO DE CAMPOS
    # Justificativa: Padronizar nomes de colunas para facilitar o JOIN com outras tabelas Silver
    .select(
        col("review_id").alias("id_avaliacao"),
        col("order_id").alias("id_pedido"),

        # REGRA: Garante que a nota seja numérica para cálculos de média (AVG)
        expr("try_cast(review_score as int)").alias("nota_avaliacao"),
        
        # REGRA: Tratamento de valores nulos para melhorar a legibilidade em ferramentas de BI
        coalesce(col("review_comment_title"), lit("Sem título")).alias("titulo_avaliacao_comentario"),
        coalesce(col("review_comment_message"), lit("Sem comentário")).alias("mensagem_avaliacao_comentario"),
        
        col("data_criacao_conv").alias("data_criacao_avaliacao"),
        col("data_resposta_conv").alias("data_resposta_avaliacao")
    )
)

# 4. CONTROLE DE AUDITORIA
df_silver_avaliacoes = df_silver_avaliacoes.withColumn("timestamp_ingestion", current_timestamp())

# 5. SALVANDO A TABELA NA CAMADA SILVER
(
    df_silver_avaliacoes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fat_avaliacoes_pedidos")
)

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, row_number, current_timestamp

# 1. LEITURA DOS DADOS
# Acessa a tabela de produtos brutos na camada Bronze
df_bronze_products = spark.read.table("bronze.tb_products")

# 2. REGRA DE NEGÓCIO: DESDUPLICAÇÃO
# Criei uma partição por 'product_id' e ordenamos pelo 'timestamp_ingestion' decrescente para 
# garantir que o registro mais recente seja o escolhido.
window_spec = Window.partitionBy("product_id").orderBy(col("timestamp_ingestion").desc())

df_silver_produtos = (
    df_bronze_products
   
    # Atribui um índice para cada versão do produto
    .withColumn("rn", row_number().over(window_spec))
   
    # Mantém apenas o registro mais atual (RN = 1)
    .filter(col("rn") == 1)
   
    # 3. SELEÇÃO E PADRONIZAÇÃO (SELECT & ALIAS)
    # Justificativa: Renomeação para termos em português e inclusão de unidades de medida 
    # (gramas/cm) para evitar erros de interpretação em cálculos logísticos.
    .select(
        col("product_id").alias("id_produto"),
        col("product_name").alias("nome_produto"),
        col("product_category_name").alias("categoria_produto"),
        col("product_name_lenght").alias("tamanho_nome_produto"),
        col("product_description_lenght").alias("tamanho_descricao_produto"),
        col("product_photos_qty").alias("quantidade_fotos"),
        col("product_weight_g").alias("peso_produto_gramas"),
        col("product_length_cm").alias("comprimento_centimetros"),
        col("product_height_cm").alias("altura_centimetros"),
        col("product_width_cm").alias("largura_centimetros")
    )
)

# 4. CONTROLE DE AUDITORIA
df_silver_produtos = df_silver_produtos.withColumn("timestamp_ingestion", current_timestamp())

# 5. CARGA NA CAMADA SILVER (LOAD)
# Justificativa: O uso do formato Delta e modo overwrite garante uma tabela 
# de dimensão sempre consistente e otimizada para leitura.
(
    df_silver_produtos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_produtos")
)

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, row_number, upper, current_timestamp

# 1. LEITURA DA ORIGEM
# Acessa os dados brutos de vendedores na camada Bronze
df_bronze_sellers = spark.read.table("bronze.tb_sellers")

# 2. CONFIGURAÇÃO DA JANELA PARA DEDUPLICAÇÃO
# Justificativa: Garante que, se houver múltiplas entradas para o mesmo vendedor, 
# utilizaremos apenas a última atualização disponível.
window_spec = Window.partitionBy("seller_id").orderBy(col("timestamp_ingestion").desc())

df_silver_vendedores = (
    df_bronze_sellers
    # Aplica numeração para identificar o registro mais recente (Lógica de Deduplicação)
    .withColumn("rn", row_number().over(window_spec))
    # Mantém apenas o registro com índice 1 (o mais atualizado)
    .filter(col("rn") == 1)
    
    # 3. MAPEAMENTO E PADRONIZAÇÃO (SELECT & ALIAS)
    # Justificativa: Tradução para português e normalização de strings para evitar 
    # inconsistências em relatórios de BI (ex: nomes de cidades em formatos mistos).
    .select(
        col("seller_id").alias("id_vendedor"),
        col("seller_name").alias("nome_vendedor"),
        col("seller_zip_code_prefix").alias("prefixo_cep"),
        # Conversão para Maiúsculas para facilitar agrupamentos geográficos
        upper(col("seller_city")).alias("cidade"),
        upper(col("seller_state")).alias("estado")
    )
)

# 4. CONTROLE DE AUDITORIA
df_silver_vendedores = df_silver_vendedores.withColumn("timestamp_ingestion", current_timestamp())

# 5. ESCRITA NA CAMADA SILVER
(
    df_silver_vendedores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_vendedores")
)

In [0]:
from pyspark.sql.functions import col, current_timestamp

# 1. LEITURA DOS DADOS
df_bronze_category_translation = spark.read.table("bronze.tb_product_category_name_translation")

# 2. TRANSFORMAÇÃO E MAPEAMENTO (SELECT & ALIAS)
# Justificativa: Renomear as colunas para um padrão mais descritivo e em português,
# explicitando qual coluna refere-se a qual idioma (PT vs EN).
df_silver_categoria_traducao = (
    df_bronze_category_translation.select(
        col("product_category_name").alias("nome_produto_pt"),
        col("product_category_name_english").alias("nome_produto_en")
    )
)

# 3. CONTROLE DE AUDITORIA
df_silver_categoria_traducao = df_silver_categoria_traducao.withColumn("timestamp_ingestion", current_timestamp())

# 4. ESCREVENDO NA CAMADA SILVER
(
    df_silver_categoria_traducao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_categoria_produtos_traducao")
)

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, explode, sequence, to_date, min, max, last, lit, expr, current_timestamp

# 1. LEITURA DA ORIGEM
df_bronze_cotacao = spark.read.table("bronze.tb_cotacao_dolar")

# 2. GERAÇÃO DE CALENDÁRIO
# Justificativa: A API do BCB não retorna dados de fins de semana. 
# Precisamos criar uma tabela de uma base com todos os dias para não haver buracos nos JOINs de vendas.
datas = df_bronze_cotacao.select(
    min(to_date(col("dataHoraCotacao"))).alias("min_dt"),
    max(to_date(col("dataHoraCotacao"))).alias("max_dt")
).collect()[0]

# Criação de um range de 1 linha e até N linhas
df_calendario = spark.range(1).select(
    explode(sequence(
        lit(datas['min_dt']), 
        lit(datas['max_dt']), 
        expr("interval 1 day")
    )).alias("data_referencia")
)

# 3. IDENTIFICAÇÃO DE LACUNAS
# Left Join: dias de fim de semana na esquerda ficarão com colunas da direita nulas.
df_completo = df_calendario.join(
    df_bronze_cotacao, 
    df_calendario.data_referencia == to_date(df_bronze_cotacao.dataHoraCotacao), 
    "left"
)

# 4. REGRA DE NEGÓCIO: FORWARD FILL
# Justificativa: Se sábado é NULL, vai pegar o valor de sexta. Se domingo é NULL, também vai pegar o de sexta.
window_spec = Window.orderBy("data_referencia").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_silver_cotacao = (
    df_completo
    .select(
        col("data_referencia").alias("data_cotacao"),
        # Usando o (ignorenulls=True) para pegar o último valor preenchido ignorando os nulos gerados pelo join.
        last(col("cotacaoCompra"), ignorenulls=True).over(window_spec).alias("valor_dolar_brl")
    )
)

# 5. CONTROLE DE AUDITORIA
df_silver_cotacao = df_silver_cotacao.withColumn("timestamp_ingestion", current_timestamp())

# 6. ESCREVENDO NA CAMADA SILVER 
(
    df_silver_cotacao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_cotacao_dolar")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, sum, round, to_date, current_timestamp, first, coalesce, lit

# 1. LEITURA DAS TABELAS SILVER
df_pedidos = spark.read.table("silver.fat_pedidos")
df_pagamentos = spark.read.table("silver.fat_pagamentos_pedidos")
df_cotacao = spark.read.table("silver.dim_cotacao_dolar")

# 2. AGREGAÇÃO DOS PAGAMENTOS
# Justificativa: Um pedido pode ter N pagamentos, precisamos do valor unico consolidado.
df_pagamentos_agrupado = (
    df_pagamentos
    .groupBy("id_pedido")
    .agg(sum("valor_pagamento_brl").alias("valor_total_brl"))
)

# 3. PREPARAÇÃO DA COTAÇÃO (FALLBACK)
# Justificativa: Precisamos evitar que o valor USD fique nulo caso a data do pedido esteja fora do range da API.
# Para isso precisamos pegar/coletar o valor mais recente disponível na tabela de cotação.
window_global = Window.orderBy(col("data_cotacao").desc())
valor_cotacao_fallback = df_cotacao.select(first("valor_dolar_brl").over(window_global)).collect()[0][0]

# 4. Fazendo uma seleção de todas as colunas necessarias para a tabela fat_silver_pedido_total
df_silver_pedido_total = (
    df_pedidos.join(df_pagamentos_agrupado, "id_pedido", "left")
    .join(
        df_cotacao, 
        to_date(col("data_compra")) == to_date(df_cotacao.data_cotacao), 
        "left"
    )
    .select(
        col("id_pedido"),
        col("id_consumidor"),
        col("status"),
        col("data_compra").alias("data_pedido"),

        # Trata nulos nos pagamentos (pedidos sem informação de valor assumem 0)
        round(coalesce(col("valor_total_brl"), lit(0)), 2).alias("valor_total_pago_brl"),

        # Cálculo de conversão com aplicação de Fallback se a cotação falhar
        round(
            coalesce(col("valor_total_brl"), lit(0)) / 
            coalesce(col("valor_dolar_brl"), lit(valor_cotacao_fallback)), 
            2
        ).alias("valor_total_pago_usd")
    )
    .withColumn("timestamp_ingestion", current_timestamp())
)

# 5. ESCREVENDO NA SILVER
(
    df_silver_pedido_total.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fat_pedido_total")
)

# --- OTIMIZAÇÃO DE PERFORMANCE ---
# Justificativa: Usando o ZORDER para melhorar a velocidade de leitura agrupando os dados relacionados nos mesmos arquivos Parquet.
spark.sql("OPTIMIZE silver.fat_pedidos ZORDER BY (id_pedido, data_compra)")
spark.sql("OPTIMIZE silver.fat_pedido_total ZORDER BY (id_pedido, data_pedido)")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,